# CP：Ulysses all-to-all 与序列/head 布局交换

## 为什么需要分析 CP 通信

FSDP 主要分片模型状态，不为一条样本分摊 attention。TP 若启用 sequence parallel，会在部分 activation 布局上切 sequence；但当前 TorchTitan Qwen3 plan 会在进入 attention/MLP 投影前把 sequence shard 转回 Replicate，因此 SP 本身不等于把完整长上下文 attention 分给多卡。

CP 的区别是让多个 rank 共同承担同一条长上下文的 attention sequence/head 工作。单条长序列的激活或 attention 成为显存、计算瓶颈时，才需要评估这种额外切分。

CP（Context Parallelism）的解决思路是**切分序列**：把一条长序列的 token 均分给 N 张卡，每张卡只保存和计算 1/N 的 token。但 self-attention 要求每个 query 看到所有 token 的 K/V——如果每个 rank 只有本地序列分片，跨 rank 的 token 将无法互相注意。

Ulysses CP 的做法不是把完整 Q/K/V 复制到每张卡，而是用 **all-to-all 交换数据所有权**：attention 前，把"本地序列 × 全部 heads"换成"全局序列 × 本地 heads"；attention 后，再逆向换回来。这两次 all-to-all 是 CP 的通信开销——理解它们的数据量、耗时和位置，才能判断长序列训练该用 CP 还是靠 FSDP 摊薄。

本节固定 Qwen3-1.7B 的 attention 张量形状和 CP degree 2，只交换 Q/K/V 形状的 buffer，不执行 attention kernel。

![CP Ulysses 的 sequence/head 交换](images/cp_collectives_zh.svg)

**图 3：** Ulysses 在 attention 前把本地 sequence × 全部 heads 换成全局 sequence × 本地 heads，attention 后再逆向交换。

## 路线

- **§1** — CP 的原理是什么？你会得到：Ulysses 的布局变换——attention 前用 all-to-all 换数据所有权，attention 后再换回来。没有 SUM 规约，只有数据重排。
- **§2** — 每个 rank 手里有什么数据？你会得到：本地 QKV buffer 33.554 MB，每 rank 发送 16.777 MB，两次 A2A 的完整数据账本。
- **§3–§4** — 在两张 NPU 上保存逐 rank 原始样本并自动生成 median/P95；再用 trace 检查 pre-attention A2A 的等待位置。
- **§5** — TorchTitan-NPU 怎么实现？你会得到：functional collective + forward pre-hook/post-hook 的注入点。
- **§6** — CP 的 all-to-all 为什么容易暴露在关键路径上？你会看到：attention 前后的布局依赖，以及如何用账本核对 A2A 次数。


## 1. CP 原理：用 all-to-all 换数据所有权，不做数值规约

### 问题：一条长上下文的 attention 如何跨卡分摊

两卡 CP 时，每个 rank 起初只保存一半 sequence（`S_local=2048`），但保存这些 token 的全部 Q/K/V heads。self-attention 中一个 query 必须读取整个上下文的 K/V；只看本地 2048 token 会改变模型语义。

TorchTitan 的 sequence parallel 也会切部分 activation 的 sequence 维，但它主要在 norm 和模块边界维持这种布局，并在 attention 投影前按 plan 做 layout redistribution。Ulysses CP 则直接为完整长上下文 attention 分摊 sequence/head 工作，两者不能混称。

### Ulysses 的解法：换布局，不复制完整 Q/K/V

Ulysses 用 all-to-all 交换数据所有权：

- **Attention 前**：`[B,S_local,全部 heads,D] → [B,S,heads/C,D]`。每个 rank 获得完整 sequence，但只负责一部分 heads。
- **Attention 后**：执行逆向 all-to-all，恢复 `[B,S_local,全部 heads,D]`。

all-to-all 只重排数据，不做 SUM。当前 NPU Ulysses 对 Q/K/V 分别交换、对 output 逆向交换；对应的自定义 autograd backward 执行逆变换。因此未融合路径的教学账本是每层 forward 4 次 A2A、backward 4 次 A2A；实际次数仍应由当前 trace 核对。

pre-attention Q/K/V 交换之后立即等待，attention kernel 必须等布局就绪才能开始，因此它是 attention 的前序依赖。

In [ ]:
B, S, H_Q, H_KV, D = 2, 4096, 16, 8, 128
cp, dtype_bytes = 2, 2
S_local = S // cp

def mb(x):
    return x / 1_000_000

q_local = B * S_local * H_Q * D * dtype_bytes
k_local = B * S_local * H_KV * D * dtype_bytes
qkv_pre = q_local + 2 * k_local

print(f'每 rank 本地 sequence：{S_local}')
print(f'all-to-all 前 Q：{mb(q_local):.3f} MB/rank')
print(f'all-to-all 前 K 或 V：{mb(k_local):.3f} MB/rank')
print(f'all-to-all 前 QKV：{mb(qkv_pre):.3f} MB/rank')
print('CP degree=2：每个 rank 向对端发送本地 Q/K/V buffer 的一半')

## 2. 数据归属：两次 all-to-all 的完整账本

Qwen3 使用 GQA：16 个 Q heads、8 个 KV heads、head dim 128。两卡 CP 各从本地序列 2048 开始。

**Attention 前的 QKV 交换**：每个 rank 的本地 Q 是 `[B, 2048, 16, 128]` = 16.777 MB，K 和 V 各 `[B, 2048, 8, 128]` = 8.389 MB，合计 33.554 MB。all-to-all 把各 buffer 沿 head 维切分发给两个 rank，再沿 sequence 维拼接。每 rank 发送本地 buffer 的一半，即 16.777 MB，接收对端同样大小的数据。

**Attention 后的输出交换**：attention 输出是 `[B, 4096, 8, 128]` = 16.777 MB（全局 sequence × 本地 Q heads）。逆向 all-to-all 切分 sequence 维、拼接 head 维，恢复为 `[B, 2048, 16, 128]`。每 rank 同样发送 8.389 MB。

关键数字：

- 每 rank 本地 QKV buffer（attention 前）：33.554 MB
- 每 rank 本地 output buffer（attention 后）：16.777 MB
- forward 每 rank 单向发送量：16.777 + 8.389 = **25.166 MB**
- 两卡 CP 的总对端交换量（4 次 A2A 的发送之和）：50.332 MB

注意这 25.166 MB 不随参数量变化——CP 的通信量由 token 数、head 数、head dim 和 dtype 决定，与模型参数量无关。这是 CP 与 FSDP 最根本的成本差异。

## 3. 在两张 NPU 上运行 all-to-all

直接运行下面的 Bash 单元。

In [ ]:
%%bash
set -euo pipefail

# 需要调度器已分配并暴露两张 NPU。
mkdir -p results
torchrun --standalone --nproc_per_node=2 scripts/cp_collectives.py \
  --output-json results/cp_latest.json \
  --profile-dir results/profiles/cp_latest

In [ ]:
# 零 overlap 串行估算的结构（从 results/cp_latest.json 读取慢 rank median）：
#
#   forward_per_layer  = q_ms + k_ms + v_ms + o_ms
#   backward_per_layer = forward_per_layer  # 仅作对称假设，需用真实 backward trace 验证
#   total_fwd_bwd      = (forward_per_layer + backward_per_layer) * 28
#
# 这不是系统最坏情况上界，也不是 step time。真实训练还包含 reshape/transpose/wait、
# 资源争用、rank skew 和可能的 overlap，可能比这个数更长或更短。

## 4. 实测结果与分析口径

脚本把 Q、K、V、output 四个独立 A2A 的全部 rank/iteration 写入 `results/cp_latest.json`，并自动报告每个 rank 与慢 rank 路径的 median、P95、range、rank spread 和有效 GB/s。`results/profiles/cp_latest/` 是另采的一轮 HCCL trace，不参与 wall-time 统计。尚未运行时不展示历史手抄数字。

比较四个 payload 时可以检查固定启动开销是否显著，但不能预设“小消息一定是某个带宽”。forward 四项 median 的和只是**零 overlap 串行估算**，不是一次融合 collective，也不是训练 step time。backward 对称、activation checkpoint 重放和 28 层累加都只是额外假设，必须用完整训练 trace 验证。

pre-attention Q/K/V A2A 后的等待是 attention 前序依赖；post-attention A2A 也必须在消费者读取输出前完成。最终训练影响应同时看 HCCL transport、wait、reshape/transpose、attention 计算、rank skew 和 profiler-off 吞吐。

## 5. TorchTitan-NPU 怎么实现 CP

`torchtitan_npu/models/qwen3/parallelize.py` 先检查 query heads 和 KV heads 能否被 CP degree 整除，然后通过 `apply_cp_to_attention_module` 把 NPU 的 CP 策略注入 attention 模块。

核心实现在 `torchtitan_npu/distributed/context_parallel/ulysses_cp.py`，使用 functional collective 和 forward pre-hook/post-hook：

- **Pre-hook**：在 attention 前对 Q/K/V 执行 all-to-all（`scatter_dim=head, gather_dim=seq`），把布局从 local-seq/all-heads 变为 global-seq/local-heads。
- **Post-hook**：在 attention 后对 output 执行逆向 all-to-all（`scatter_dim=seq, gather_dim=head`），恢复为 local-seq/all-heads。
- **Backward**：`AllToAll` 是一个自定义 `torch.autograd.Function`，其 `backward` 交换 scatter/gather 维度，对输出梯度执行逆向 all-to-all——因此反向通信不需要额外手写。

CP 的通信量由 token 数、head 数、head dim、dtype 和 CP degree 决定，与参数量无关——这是分析 CP 成本的核心公式锚点。

### 计算思考

如果序列长度从 4,096 改为 8,192，而其他 Qwen3 维度保持不变，请重新计算 attention 前的 QKV payload。哪个数字会翻倍？哪条数据归属关系保持不变？

## 6. CP 通信的暴露时间

Ulysses CP 在 attention 前后各做一次布局交换。下面的账本只用于核对 A2A 的调用次数和 payload，不把通信字节数直接当成训练 step time。

attention 前的 A2A 是硬依赖：attention kernel 必须等布局变成“全局序列 × 本地 heads”后才能开始，因此不能简单地像 FSDP 参数预取那样提前到任意位置。attention 后的逆向 A2A 也必须在后续算子读取输出前完成。只有与当前布局无关的计算，才可能与 A2A 重叠；实际能隐藏多少，要看 trace 中的 `wait` 和拓扑。


### all_to_all_single 的实现细节

`AllToAll` 是自定义 `torch.autograd.Function`：`forward` 调用 functional collective 后在 `wait_tensor` 处等待布局转换完成，`backward` 交换 scatter/gather 维度对输出梯度执行逆向 all-to-all。这意味着 pre-attention A2A 的 `wait_tensor` 是 attention 的前序依赖——不能声称这部分通信被 attention 计算隐藏。CP 的反向通信不需要额外手写，是前向布局变换的自动求导逆变换。

### 28 层通信账本（代码验证）

代入实测参数，用公式 $V_{CP, forward} = \frac{N-1}{N} \cdot (M_{QKV} + M_O)$ 计算 28 层完整训练的 all-to-all 通信量。

In [ ]:
LAYERS = 28

def gb(x):
    return x / 1_000_000_000

# attention output 的元素数与 Q 对应；在 CP 交换前后总字节数不变。
output_local = B * S_local * H_Q * D * dtype_bytes
remote_fraction = (cp - 1) / cp
forward_send_per_layer = remote_fraction * (qkv_pre + output_local)
forward_send = LAYERS * forward_send_per_layer
backward_send = forward_send
train_send = forward_send + backward_send

# 若 checkpoint 重放完整 attention forward，会再支付一遍 forward A2A。
train_send_with_full_forward_recompute = train_send + forward_send

print(f'每层 forward：QKV 本地 buffer {mb(qkv_pre):.3f} MB，输出 buffer {mb(output_local):.3f} MB')
print(f'每层 forward 每 rank 发送：{mb(forward_send_per_layer):.3f} MB')
print(f'28 层 forward：最多 {4 * LAYERS} 次 A2A，发送 {gb(forward_send):.3f} GB/rank')
print(f'28 层 backward：最多 {4 * LAYERS} 次逆向 A2A，发送 {gb(backward_send):.3f} GB/rank')
print(f'一次 forward+backward：最多 {8 * LAYERS} 次 A2A')
print(f'  单向发送：{gb(train_send):.3f} GB/rank')
print(f'  单向接收：{gb(train_send):.3f} GB/rank')
print(f'  发送+接收：{gb(2 * train_send):.3f} GB/rank')
print(f'若完整重算 attention forward：单向发送约 {gb(train_send_with_full_forward_recompute):.3f} GB/rank')

代码输出的是 CP 的字节与调用次数账本。当前机器的 latency 必须从 `results/cp_latest.json` 读取；pre-attention 与 output 的暴露比例必须从双 rank trace 判断。

CP 适用于需要为一条长上下文分摊 attention/activation 的情况。它是否值得采用，取决于目标序列长度下节省的计算/显存能否覆盖 A2A、布局变换和等待；最终选择应以相同 workload 的端到端 trace 和 profiler-off 吞吐验证。


## 练习：选择与判断

1. （判断题）Ulysses CP 的 all-to-all 主要交换 sequence/head 数据归属，不执行 SUM 规约。

2. （单选题）固定 B=2、全局 S=4096、Q heads=16、CP=2 时，Q 在 pre AllToAll 后的逻辑 shape 是什么？
    A. [2,2048,16,128]
    B. [2,4096,8,128]
    C. [2,4096,16,128]
    D. [2,2048,8,128]

3. （判断题）启用 sequence parallel 后，TP 也可能在部分 activation layout 上切 sequence，因此不能绝对地说“TP 不切 sequence”。

4. （判断题）CP 支持更长上下文，所以在任何 shape 和 workload 下都必然比 FSDP 更快。

In [ ]:
!cat ./answer/07.04_answer.txt
